# 5-2 Token単位による文章分類
同じ10件を学習し、Janomeで日本語をTokenに分割して推論します。


In [ ]:
training_data = [('沖縄の海でシュノーケリングを楽しみたい', 1), ('京都まで新幹線で行き寺を見学する', 1), ('ホテルを予約して北海道を三日間観光する', 1), ('成田からパリへの航空券を探している', 1), ('温泉旅館に泊まり箱根を散策したい', 1), ('Pythonでファイルを読み込む方法を調べる', 0), ('今日の夕食はカレーを作る予定だ', 0), ('会社の会議資料を明日までに作成する', 0), ('野球の試合結果をニュースで確認した', 0), ('新しいパソコンのメモリを増設したい', 0)]
samples = ['来月、飛行機で沖縄へ行く', '京都の写真をパソコンに保存する', '駅まで歩いて会社へ行く']


In [ ]:
import math

def train_model(data, split_func):
    class_count = [0, 0]
    item_count = [{}, {}]
    total = [0, 0]
    vocabulary = set()

    for text, label in data:
        class_count[label] += 1
        for item in split_func(text):
            vocabulary.add(item)
            item_count[label][item] = item_count[label].get(item, 0) + 1
            total[label] += 1

    return class_count, item_count, total, vocabulary

def predict(text, model, split_func):
    class_count, item_count, total, vocabulary = model
    scores = []

    for label in [0, 1]:
        score = math.log(class_count[label] / sum(class_count))
        for item in split_func(text):
            score += math.log(
                (item_count[label].get(item, 0) + 1) /
                (total[label] + len(vocabulary))
            )
        scores.append(score)

    return (1 if scores[1] > scores[0] else 0), scores


In [ ]:
!pip -q install janome

from janome.tokenizer import Tokenizer

tokenizer = Tokenizer()

def split_text(text):
    return [
        token.surface
        for token in tokenizer.tokenize(text)
        if token.part_of_speech.split(",")[0] != "記号"
    ]

SPLIT_NAME = "Token"
model = train_model(training_data, split_text)


## 任意の文章を判定
3つのサンプルを選択できます。文章欄を書き換えて自由に試すこともできます。


In [ ]:
import ipywidgets as widgets
from IPython.display import display

sample_select = widgets.Dropdown(
    options=samples,
    value=samples[0],
    description="サンプル:"
)

text_input = widgets.Textarea(
    value=samples[0],
    description="文章:",
    layout=widgets.Layout(width="750px", height="90px")
)

judge_button = widgets.Button(description="判定")
output = widgets.Output()

def sample_changed(change):
    text_input.value = change["new"]

sample_select.observe(sample_changed, names="value")

def on_judge(_):
    with output:
        output.clear_output()
        items = split_text(text_input.value)
        answer, scores = predict(text_input.value, model, split_text)

        print(SPLIT_NAME, ":", " / ".join(items))
        print()
        print("旅行以外 :", round(scores[0], 2))
        print("旅行     :", round(scores[1], 2))
        print()
        print("判定 :", "旅行" if answer else "旅行以外")

judge_button.on_click(on_judge)

display(sample_select, text_input, judge_button, output)
